In [ ]:
# """
# Train a ResNet-50 from scratch (randomly initialized, NOT pretrained) on
# Imagenette, where image paths and labels come from a CSV file in /Data
# instead of a folder-per-class structure.

# Uses ONLY `torch` + `torchvision` (+ Python's built-in `csv` module) — no
# Hugging Face libraries at all.

# EXPECTED CSV FORMAT
# --------------------
# /Data/<something>.csv, with:
#   - column 1: image path (assumed relative to /Data, e.g. "train/123.jpg")
#   - column 2: label (a string class name, e.g. "cat")

#     path,label
#     train/n01440764/img1.jpeg,tench
#     train/n01440764/img2.jpeg,tench
#     val/n02979186/img1.jpeg,cassette_player
#     ...

# ASSUMPTIONS (change these if your CSV differs):
#   - The CSV has a header row (set HAS_HEADER = False below if not).
#   - There's no separate train/val column, so this script creates its own
#     random train/validation split from the CSV rows.
#   - Paths in the CSV are relative to /Data. If they're already absolute,
#     the code below still works unchanged (os.path.join handles both).

In [ ]:
import csv
import os
import random
 
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm   # progress bar for the training loop

In [ ]:
%cd ..


%pwd

In [ ]:
# ---------------------------------------------------------------------------
# 0. CONFIG — adjust these to match your CSV
# ---------------------------------------------------------------------------

DATA_DIR = "/users/gcb25209/Data/imagenette2"
CSV_PATH = "/users/gcb25209/Data/imagenette2/noisy_imagenette.csv"   # <-- change to your actual CSV filename
HAS_HEADER = True
VAL_FRACTION = 0.2              # 20% of rows held out for validation
RANDOM_SEED = 42

# --- Adversarial validation set (the output folder of the MI-FGSM notebook) ---
# Use ABSOLUTE paths here: the `%cd ..` cell above changes the working directory,
# so relative paths can silently point somewhere different each time it is run.
ADV_DIR = "/users/gcb25209/Data/adv_imagenette"            # <-- CHANGE to your MI-FGSM output folder
ADV_METADATA_PATH = os.path.join(ADV_DIR, "metadata.csv")  # written by the MI-FGSM notebook

# --- How much of the validation set to switch to adversarial images ---
# Percentage (0-100) of the validation images replaced by their MI-FGSM adversarial
# version; the rest stay clean. Typical choices: 5, 10, 25, 50, 100.
# (0 = fully clean, which is handy as a baseline on exactly the same images.)
ADV_PERCENT = 25
# Seed controlling WHICH images are swapped. With a fixed seed the choice is
# reproducible, and smaller percentages are always subsets of larger ones
# (e.g. the 5% swapped images are all among the 10% swapped images).
ADV_SWAP_SEED = 0

# --- Training hyperparameters (moved here from the training cell so that the single
#     run in step 4 and the sweep in step 9 share exactly the same settings) ---
EPOCHS = 20
LEARNING_RATE = 1e-3
TRAIN_SEED = 42                 # seed for model initialisation + training-data order (used by the sweep)

# --- Maximum-accuracy reporting ---
N_EPOCHS_TO_CHECK = 15          # report the best validation accuracy within this many initial epochs

# --- SWEEP: run a NEW experiment for each adversarial percentage, one after another ---
RUN_SWEEP = False                       # set to True to enable step 9 (it does nothing while False)
SWEEP_PERCENTS = [5, 10, 25, 50, 100]   # percentages to run, in this order (0 = clean baseline is allowed)
# True : every run uses TRAIN_SEED, so training is the same in each run and ONLY the validation
#        mix differs (isolates the effect of the adversarial percentage).
# False: run k uses TRAIN_SEED + k, so runs are independent and also show run-to-run variation.
SWEEP_SAME_SEED = True

In [ ]:
# ---------------------------------------------------------------------------
# 1. READ THE CSV AND BUILD (path, label) PAIRS
# ---------------------------------------------------------------------------
def load_csv_rows(csv_path, has_header):
    """Read the CSV and return a list of (image_path, label_str) tuples."""
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        if has_header:
            next(reader)  # skip header row
        for row in reader:
            image_path, label = row[0], row[1]
            rows.append((image_path, label))
    return rows


all_rows = load_csv_rows(CSV_PATH, HAS_HEADER)

# Build a sorted list of unique class names, then a name -> integer index
# mapping (models need integer labels, not strings).
class_names = sorted({label for _, label in all_rows})
label_to_index = {name: i for i, name in enumerate(class_names)}
num_labels = len(class_names)
print(f"Found {num_labels} classes: {class_names}")

# Shuffle deterministically, then split into train/val since the CSV
# itself doesn't mark a split.
random.seed(RANDOM_SEED)
random.shuffle(all_rows)

split_index = int(len(all_rows) * (1 - VAL_FRACTION))
train_rows = all_rows[:split_index]
val_rows = all_rows[split_index:]
print(f"Train rows: {len(train_rows)} | Val rows: {len(val_rows)}")



In [ ]:
# ---------------------------------------------------------------------------
# 1b. PAIR EACH VALIDATION IMAGE WITH ITS MI-FGSM ADVERSARIAL VERSION
# ---------------------------------------------------------------------------
# The train/val SPLIT above is unchanged: `train_rows` still points at the clean
# images, and `val_rows` still decides WHICH images are held out (same seed ->
# same split). This step finds, for each held-out image, the adversarial image
# that the MI-FGSM notebook crafted from it. Step 1c then decides how many of
# those pairs are actually switched to the adversarial version (ADV_PERCENT).
#
# HOW THE PAIRING WORKS
# The MI-FGSM notebook wrote a metadata.csv whose `source_path` column records
# which original image each adversarial PNG was made from. We match each
# validation row to its metadata record by FILE STEM (the filename without its
# folder or extension, e.g. "n01440764_10026"). Matching on the stem rather than
# the full path is deliberate: the full path may have been written differently
# when the adversarial set was generated (relative vs absolute, different mount
# point), and Imagenette filenames are unique so the stem is a reliable key.
# We keep each row's OWN label from the CSV, so labels stay consistent with
# `label_to_index` used for training.
#
# WHY SOME VALIDATION IMAGES ARE DROPPED
# The MI-FGSM notebook only attacks images that its pretrained models classified
# correctly, so some originals have no adversarial counterpart. Those rows are
# left out of the validation set entirely (and counted below). That way EVERY
# image in the validation set has both a clean and an adversarial version, so the
# underlying images are identical whatever ADV_PERCENT you choose, and results at
# different percentages are directly comparable.
from collections import Counter


def file_stem(path):
    # "train/n01440764/n01440764_10026.JPEG" -> "n01440764_10026"
    return os.path.splitext(os.path.basename(path))[0].lower()


# --- read the generator's metadata: stem -> record -------------------------
adv_by_stem = {}
ambiguous_stems = set()          # stems that appear more than once (can't pair safely)
with open(ADV_METADATA_PATH, newline="") as f:
    for record in csv.DictReader(f):
        stem = file_stem(record["source_path"])
        if stem in adv_by_stem:
            ambiguous_stems.add(stem)
        adv_by_stem[stem] = record

# Stems that appear more than once among our own validation rows are also unsafe.
val_stem_counts = Counter(file_stem(p) for p, _ in val_rows)

# --- build the paired list: (clean path rel. to DATA_DIR, adversarial path rel. to ADV_DIR, label) ---
paired_rows = []
label_to_adv_folders = {}        # consistency check: each label should map to ONE class folder
n_not_found = 0                  # no adversarial counterpart (e.g. attack skipped this image)
n_ambiguous = 0
for image_path, label_str in val_rows:
    stem = file_stem(image_path)
    if stem in ambiguous_stems or val_stem_counts[stem] > 1:
        n_ambiguous += 1
        continue
    record = adv_by_stem.get(stem)
    if record is None:
        n_not_found += 1
        continue
    paired_rows.append((image_path, record["file"], label_str))   # keep the CSV's own label
    label_to_adv_folders.setdefault(label_str, set()).add(record["class"])

print(f"Original validation rows:            {len(val_rows)}")
print(f"  paired with an adversarial image:  {len(paired_rows)}")
print(f"  no adversarial counterpart:        {n_not_found}  (dropped)")
print(f"  ambiguous filename:                {n_ambiguous}  (dropped)")

assert len(paired_rows) > 0, (
    "No validation images could be paired. Check ADV_DIR / ADV_METADATA_PATH and "
    "that the MI-FGSM notebook was run on the same Imagenette images."
)
if len(paired_rows) < 0.5 * len(val_rows):
    print("WARNING: fewer than half of the validation rows were paired. "
          "Was the MI-FGSM notebook run on the full CSV (or MAX_IMAGES set)?")

# --- sanity checks ----------------------------------------------------------
# 1) Every label must map to exactly one adversarial class folder; otherwise the
#    pairing (or the label column) is wrong and accuracy numbers would be meaningless.
inconsistent = {k: sorted(v) for k, v in label_to_adv_folders.items() if len(v) > 1}
if inconsistent:
    raise ValueError(f"Labels map to more than one adversarial class folder: {inconsistent}")
print("\nLabel -> adversarial folder:")
for label_str in sorted(label_to_adv_folders):
    print(f"  {label_str:<25s} -> {next(iter(label_to_adv_folders[label_str]))}")

# 2) All adversarial files exist.
missing_files = [adv for _, adv, _ in paired_rows if not os.path.exists(os.path.join(ADV_DIR, adv))]
assert not missing_files, f"{len(missing_files)} adversarial files missing, e.g. {missing_files[0]}"

# 3) Adversarial images are already 224x224 (they are NOT resized when loaded).
with Image.open(os.path.join(ADV_DIR, paired_rows[0][1])) as im:
    assert im.size == (224, 224), f"Expected 224x224 adversarial images, got {im.size}"

# 4) Class balance of the paired validation set.
paired_counts = Counter(label for _, _, label in paired_rows)
print("\nPaired validation images per class:", {k: paired_counts[k] for k in sorted(paired_counts)})
print("Example pair:", paired_rows[0][0], "->", paired_rows[0][1])

In [ ]:
# ---------------------------------------------------------------------------
# 1c. CHOOSE HOW MUCH OF THE VALIDATION SET TO SWITCH TO ADVERSARIAL IMAGES
# ---------------------------------------------------------------------------
# ADV_PERCENT (set in the config cell) is the percentage of the paired validation
# images that are replaced by their adversarial version; the remainder stay clean.
# The size of the validation set does not change, only the clean/adversarial mix.
#
# The swapped images are chosen WITHIN EACH CLASS (stratified), so every class
# contributes the same percentage of adversarial images and the adversarial part
# is not accidentally dominated by one class. Within each class we shuffle once
# with a fixed seed and take the first N, which makes the choice reproducible and
# nested: the images swapped at 5% are a subset of those swapped at 10%, 25%, ...
#
# This is written as a function so the sweep (step 9) can reuse it for every
# percentage; the lines after it apply it to ADV_PERCENT for the single run.
def select_val_rows(adv_percent):
    # Returns (rows, swapped_per_class). Each row is
    # (full path to the image file, label string, is_adversarial).
    assert 0 <= adv_percent <= 100, f"adversarial percentage must be between 0 and 100, got {adv_percent}"

    swap_rng = random.Random(ADV_SWAP_SEED)   # local RNG: leaves the global random state untouched

    rows_by_label = {}
    for row in paired_rows:
        rows_by_label.setdefault(row[2], []).append(row)

    mixed_rows = []
    swapped_per_class = {}
    for label_str in sorted(rows_by_label):
        class_rows = sorted(rows_by_label[label_str])   # sort first so the shuffle is deterministic
        swap_rng.shuffle(class_rows)
        n_swap = round(len(class_rows) * adv_percent / 100)
        swapped_per_class[label_str] = (n_swap, len(class_rows))
        for i, (clean_rel, adv_rel, label) in enumerate(class_rows):
            if i < n_swap:
                mixed_rows.append((os.path.join(ADV_DIR, adv_rel), label, True))      # adversarial
            else:
                mixed_rows.append((os.path.join(DATA_DIR, clean_rel), label, False))   # clean
    return mixed_rows, swapped_per_class


val_mixed_rows, swapped_per_class = select_val_rows(ADV_PERCENT)

n_adv_total = sum(is_adv for _, _, is_adv in val_mixed_rows)
print(f"Validation set: {len(val_mixed_rows)} images | "
      f"{n_adv_total} adversarial ({100 * n_adv_total / len(val_mixed_rows):.1f}%, "
      f"requested {ADV_PERCENT}%) | {len(val_mixed_rows) - n_adv_total} clean")
print("Adversarial / total per class:",
      {k: f"{a}/{t}" for k, (a, t) in swapped_per_class.items()})

In [ ]:
# ---------------------------------------------------------------------------
# 2. A CUSTOM DATASET THAT READS IMAGES USING THE CSV ROWS
# ---------------------------------------------------------------------------
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),                 # PIL image -> [0,1] tensor, shape (C,H,W)
    transforms.Normalize(mean=MEAN, std=STD),
])


class CsvImageDataset(Dataset):
#     """
#     A minimal PyTorch Dataset that, for each CSV row, loads the image at
#     `data_dir/path` from disk and returns (transformed_image, label_index).

#     This replaces `torchvision.datasets.ImageFolder`, which expected a
#     folder-per-class layout — here the CSV tells us the label directly, so
#     we don't need any particular folder structure.
#     """

    def __init__(self, rows, data_dir, label_to_index, transform):
        self.rows = rows
        self.data_dir = data_dir
        self.label_to_index = label_to_index
        self.transform = transform

    def __len__(self):
        # Required so DataLoader knows how many samples exist.
        return len(self.rows)

    def __getitem__(self, idx):
        # Required so DataLoader can fetch a single (image, label) sample.
        image_path, label_str = self.rows[idx]
        full_path = os.path.join(self.data_dir, image_path)

        image = Image.open(full_path).convert("RGB")  # force 3 channels
        image = self.transform(image)

        label = self.label_to_index[label_str]
        return image, label


train_data = CsvImageDataset(train_rows, DATA_DIR, label_to_index, transform)
# VALIDATION: a mix of clean and MI-FGSM adversarial images (chosen in step 1c).
# The two kinds need different preprocessing:
#   - clean images use the same `transform` as training (resize to 224x224);
#   - adversarial images were already resized + center-cropped to 224x224 and saved
#     as lossless PNGs, so they must NOT be resized / cropped / re-compressed again
#     (that would alter the perturbation). We only convert to a tensor and apply the
#     SAME normalization as training, so the model sees inputs on the same scale.
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


class MixedValDataset(Dataset):
    # Like CsvImageDataset, but every row already carries the FULL path to its
    # image file plus a flag saying whether it is adversarial. The flag selects
    # which preprocessing is applied.
    def __init__(self, rows, label_to_index, clean_transform, adv_transform):
        self.rows = rows
        self.label_to_index = label_to_index
        self.clean_transform = clean_transform
        self.adv_transform = adv_transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        full_path, label_str, is_adversarial = self.rows[idx]
        image = Image.open(full_path).convert("RGB")
        tf = self.adv_transform if is_adversarial else self.clean_transform
        return tf(image), self.label_to_index[label_str]


val_data = MixedValDataset(val_mixed_rows, label_to_index, transform, val_transform)
n_adv = sum(is_adv for _, _, is_adv in val_mixed_rows)
print(f"Validation set: {len(val_data)} images | {n_adv} adversarial "
      f"({100 * n_adv / len(val_data):.1f}%) | {len(val_data) - n_adv} clean")

BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)



In [ ]:
# ---------------------------------------------------------------------------
# 3. BUILD A BLANK (UNTRAINED) RESNET-50
# ---------------------------------------------------------------------------
# `weights=None` gives the standard ResNet-50 architecture with fresh,
# randomly-initialized weights (as opposed to `weights="IMAGENET1K_V2"`,
# which would download pretrained weights).
model = models.resnet50(weights=None)

# Replace the final layer (sized for ImageNet's 1000 classes) with one
# sized for our number of classes from the CSV.
model.fc = nn.Linear(model.fc.in_features, num_labels)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
# ---------------------------------------------------------------------------
# 4. PLAIN PYTORCH TRAINING LOOP
# ---------------------------------------------------------------------------
# EPOCHS and LEARNING_RATE are now set in the config cell at the top, so that the
# sweep in step 9 uses exactly the same values.
 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()
 
# Lists to record accuracy after every epoch, so we can plot them once
# training finishes (see step 6 below).
train_accuracies = []
val_accuracies = []
 
 
def run_validation():
    """Evaluate the model on the val set and return (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
 
            logits = model(images)
            loss = loss_fn(logits, targets)
            total_loss += loss.item() * images.size(0)
 
            predictions = logits.argmax(dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)
 
    return total_loss / total, correct / total
 
 
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
 
    # `tqdm(...)` wraps the DataLoader and draws a live progress bar that
    # advances one step per batch, showing how far through the epoch we
    # are and how fast batches are being processed (it/s). `set_postfix`
    # adds the running average loss to the end of the bar so you can watch
    # it drop as training progresses.
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}", unit="batch")
 
    samples_seen = 0
    for images, targets in progress_bar:
        images, targets = images.to(device), targets.to(device)
 
        optimizer.zero_grad()
        logits = model(images)                # forward pass
        loss = loss_fn(logits, targets)       # compare predictions to true labels
        loss.backward()                        # backpropagate gradients
        optimizer.step()                        # update weights
 
        running_loss += loss.item() * images.size(0)
        samples_seen += images.size(0)
 
        # Track how many training predictions were correct, so we can
        # report a training accuracy alongside the loss (used for the
        # accuracy-over-time plot in step 6).
        predictions = logits.argmax(dim=1)
        correct_train += (predictions == targets).sum().item()
        total_train += targets.size(0)
 
        # Update the bar's live display with the running average loss so far.
        progress_bar.set_postfix(loss=running_loss / samples_seen)
 
    train_loss = running_loss / len(train_data)
    train_acc = correct_train / total_train
    val_loss, val_acc = run_validation()
 
    # Record this epoch's accuracies so they can be plotted after training.
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
 
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# 5. SAVE THE TRAINED MODEL
# ---------------------------------------------------------------------------
torch.save(model.state_dict(), "./resnet50_scratch_imagenette.pt")
print("Model weights saved to ./resnet50_scratch_imagenette.pt")

In [ ]:
# ---------------------------------------------------------------------------
# 6. PLOT ACCURACY OVER TIME
# ---------------------------------------------------------------------------
# `train_accuracies` and `val_accuracies` were filled in with one value per
# epoch inside the training loop above. Plotting both on the same axes
# makes it easy to see the model improving over time, and to spot
# overfitting (train accuracy climbing while val accuracy flattens/drops).
import matplotlib.pyplot as plt
 
epochs_range = range(1, EPOCHS + 1)
 
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_accuracies, marker="o", label="Train accuracy")
plt.plot(epochs_range, val_accuracies, marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over time")
plt.legend()
plt.grid(True)
 
plot_path = "./accuracy_over_time.png"
plt.savefig(plot_path)
print(f"Accuracy plot saved to {plot_path}")
 
# `plt.show()` opens an interactive window if you're running this in an
# environment with a display (e.g. locally, not over SSH with no X server).
# It's harmless to leave in even if there's no display — some backends
# just no-op in that case — but comment it out if it ever hangs your run.
plt.show()
 

In [ ]:
# ---------------------------------------------------------------------------
# 7. DISPLAY A FEW RANDOM TRAINING IMAGES WITH THEIR LABELS
# ---------------------------------------------------------------------------
# Purely a sanity check — lets you eyeball that images and labels are being
# read from the CSV correctly. We pull straight from `train_data`, so each
# image comes back already resized + normalized by `transform`.
NUM_SAMPLES_TO_SHOW = 6
sample_indices = random.sample(range(len(train_data)), NUM_SAMPLES_TO_SHOW)
 
fig, axes = plt.subplots(1, NUM_SAMPLES_TO_SHOW, figsize=(3 * NUM_SAMPLES_TO_SHOW, 3))
 
mean_tensor = torch.tensor(MEAN).view(3, 1, 1)
std_tensor = torch.tensor(STD).view(3, 1, 1)
 
for ax, idx in zip(axes, sample_indices):
    image_tensor, label_index = train_data[idx]
 
    # The transform normalized pixel values earlier (subtract mean, divide
    # by std) so the model trains well — but that also makes the image look
    # washed-out/wrong if plotted directly. Undo that here purely for
    # display: multiply back by std and add back the mean.
    unnormalized = image_tensor * std_tensor + mean_tensor
    unnormalized = unnormalized.clamp(0, 1)  # keep valid pixel range after undoing normalization
 
    # matplotlib expects images as (height, width, channels), but PyTorch
    # tensors are (channels, height, width) — permute reorders the axes.
    image_for_display = unnormalized.permute(1, 2, 0).numpy()
 
    ax.imshow(image_for_display)
    ax.set_title(class_names[label_index])
    ax.axis("off")
 
plt.tight_layout()
 
sample_plot_path = "./sample_training_images.png"
plt.savefig(sample_plot_path)
print(f"Sample training images saved to {sample_plot_path}")
 
plt.show()
 

In [ ]:
# ---------------------------------------------------------------------------
# 8. MAXIMUM VALIDATION ACCURACY OVER THE FIRST 15 EPOCHS
# ---------------------------------------------------------------------------
# `val_accuracies` holds one value per epoch (index 0 = epoch 1), recorded in the
# training loop. Note that the validation set is now a mix of clean and MI-FGSM
# adversarial images (the share is set by ADV_PERCENT in the config cell).
# N_EPOCHS_TO_CHECK (set in the config cell, default 15) says how many initial
# epochs to consider. Slicing with [:N_EPOCHS_TO_CHECK] keeps only those; if fewer
# epochs were run, the slice simply uses however many exist.

first_n = val_accuracies[:N_EPOCHS_TO_CHECK]
max_val_acc = max(first_n)
best_epoch = first_n.index(max_val_acc) + 1     # +1 because epochs are numbered from 1

print(f"Maximum validation accuracy over the first {len(first_n)} epochs: "
      f"{max_val_acc:.4f} ({max_val_acc * 100:.2f}%) at epoch {best_epoch} "
      f"[validation set: {ADV_PERCENT}% adversarial]")

In [ ]:
# ---------------------------------------------------------------------------
# 9. SWEEP: RUN A NEW EXPERIMENT FOR EACH ADVERSARIAL PERCENTAGE, IN SEQUENCE
# ---------------------------------------------------------------------------
# For every value in SWEEP_PERCENTS (e.g. 5, 10, 25, 50, 100) this cell:
#   1. builds a validation set in which that percentage of images is switched to
#      its MI-FGSM adversarial version (same selection as step 1c),
#   2. trains a brand-new, randomly initialised ResNet-50 on the CLEAN training
#      set, using the same recipe, EPOCHS and LEARNING_RATE as step 4,
#   3. records train/validation loss and accuracy after every epoch.
# The experiments run one after another. Results are written to sweep_results.csv
# after EACH experiment, so nothing is lost if the notebook is interrupted.
#
# Notes
#   - Controlled by RUN_SWEEP, SWEEP_PERCENTS and SWEEP_SAME_SEED in the config
#     cell. While RUN_SWEEP is False this cell does nothing.
#   - The training set is identical in every run; only the validation mix changes.
#     Each experiment is a full training run, so total time is roughly
#     len(SWEEP_PERCENTS) x the time of the single run.
#   - Models trained in the sweep are not saved, only their accuracy history.
#   - To run ONLY the sweep (skipping the single run in steps 3-6 and 8), run the
#     cells from the top down to the dataset cell (step 2), then this cell and the
#     summary cell after it.
import time

SWEEP_RESULTS_CSV = "./sweep_results.csv"
sweep_loss_fn = nn.CrossEntropyLoss()


def evaluate(eval_model, loader):
    # Return (average loss, accuracy) of `eval_model` over `loader`.
    eval_model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            logits = eval_model(images)
            total_loss += sweep_loss_fn(logits, targets).item() * images.size(0)
            correct += (logits.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)
    return total_loss / total, correct / total


def run_one_experiment(adv_percent, seed):
    # Train a fresh ResNet-50 and validate on a set with `adv_percent`% adversarial images.
    # Returns (per-epoch history, number of adversarial val images, total val images).
    torch.manual_seed(seed)   # fixes model initialisation and training-data order for this run

    # Validation set for this percentage (training data/loader are reused unchanged).
    rows, _ = select_val_rows(adv_percent)
    n_adv = sum(is_adv for _, _, is_adv in rows)
    exp_val_loader = DataLoader(
        MixedValDataset(rows, label_to_index, transform, val_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    )

    # Fresh, randomly initialised ResNet-50 (same recipe as step 3).
    exp_model = models.resnet50(weights=None)
    exp_model.fc = nn.Linear(exp_model.fc.in_features, num_labels)
    exp_model.to(device)
    exp_optimizer = optim.Adam(exp_model.parameters(), lr=LEARNING_RATE)

    history = []
    for epoch in range(EPOCHS):
        exp_model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        progress = tqdm(train_loader, desc=f"[{adv_percent}% adv] Epoch {epoch + 1}/{EPOCHS}",
                        unit="batch", leave=False)
        for images, targets in progress:
            images, targets = images.to(device), targets.to(device)
            exp_optimizer.zero_grad()
            logits = exp_model(images)
            loss = sweep_loss_fn(logits, targets)
            loss.backward()
            exp_optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct_train += (logits.argmax(dim=1) == targets).sum().item()
            total_train += targets.size(0)
            progress.set_postfix(loss=running_loss / total_train)

        train_loss, train_acc = running_loss / total_train, correct_train / total_train
        val_loss, val_acc = evaluate(exp_model, exp_val_loader)
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{adv_percent}% adv] Epoch {epoch + 1}/{EPOCHS} | "
              f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")
    return history, n_adv, len(rows)


sweep_history = {}   # adversarial percent -> list of per-epoch dicts
sweep_meta = {}      # adversarial percent -> (n adversarial val images, n val images)


def save_sweep_results():
    # Rewrite the CSV with everything finished so far (called after every experiment).
    with open(SWEEP_RESULTS_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["adv_percent", "epoch", "train_loss", "train_acc", "val_loss", "val_acc",
                         "n_val_adversarial", "n_val_total"])
        for pct, hist in sweep_history.items():
            n_adv, n_val = sweep_meta[pct]
            for h in hist:
                writer.writerow([pct, h["epoch"], f"{h['train_loss']:.6f}", f"{h['train_acc']:.6f}",
                                 f"{h['val_loss']:.6f}", f"{h['val_acc']:.6f}", n_adv, n_val])


if not RUN_SWEEP:
    print("RUN_SWEEP is False, so the sweep was skipped. Set RUN_SWEEP = True in the config cell to run it.")
else:
    assert len(set(SWEEP_PERCENTS)) == len(SWEEP_PERCENTS), "SWEEP_PERCENTS contains duplicates"
    assert all(0 <= p <= 100 for p in SWEEP_PERCENTS), "every SWEEP_PERCENTS value must be between 0 and 100"

    print(f"Running {len(SWEEP_PERCENTS)} experiments x {EPOCHS} epochs: {SWEEP_PERCENTS}")
    if SWEEP_SAME_SEED:
        print(f"Seed: {TRAIN_SEED} for every run -> training is the same each time (up to GPU "
              f"non-determinism); only the validation mix differs.")
    else:
        print(f"Seeds: {TRAIN_SEED} + run index -> independent runs.")

    sweep_t0 = time.time()
    for run_idx, pct in enumerate(SWEEP_PERCENTS):
        seed = TRAIN_SEED if SWEEP_SAME_SEED else TRAIN_SEED + run_idx
        print(f"\n===== Experiment {run_idx + 1}/{len(SWEEP_PERCENTS)}: "
              f"{pct}% adversarial validation (seed {seed}) =====")
        run_t0 = time.time()
        history, n_adv, n_val = run_one_experiment(pct, seed)
        sweep_history[pct] = history
        sweep_meta[pct] = (n_adv, n_val)
        save_sweep_results()    # checkpoint results after every experiment
        print(f"Finished {pct}% in {(time.time() - run_t0) / 60:.1f} min "
              f"({n_adv}/{n_val} validation images adversarial). Results saved to {SWEEP_RESULTS_CSV}")

    print(f"\nSweep complete in {(time.time() - sweep_t0) / 60:.1f} min.")

In [ ]:
# ---------------------------------------------------------------------------
# 10. SWEEP SUMMARY: TABLE AND PLOTS
# ---------------------------------------------------------------------------
# Left plot : validation accuracy per epoch, one line per adversarial percentage.
# Right plot: best validation accuracy within the first N_EPOCHS_TO_CHECK epochs
#             against the percentage of adversarial validation images.
# "Validation accuracy" is measured on a set that is X% adversarial, so it should
# fall as X grows if the adversarial images transfer to the model being trained.
import matplotlib.pyplot as plt

if not globals().get("sweep_history"):
    print("No sweep results yet. Set RUN_SWEEP = True in the config cell and run step 9 first.")
else:
    summary = []
    for pct in sorted(sweep_history):
        accs = [h["val_acc"] for h in sweep_history[pct]]
        first_n = accs[:N_EPOCHS_TO_CHECK]
        best_acc = max(first_n)
        summary.append((pct, sweep_meta[pct], accs[-1], best_acc, first_n.index(best_acc) + 1))

    print(f"{'adv %':>6} | {'adv / total val':>16} | {'final val acc':>13} | "
          f"{'max val acc (first ' + str(N_EPOCHS_TO_CHECK) + ')':>22} | {'best epoch':>10}")
    print("-" * 80)
    for pct, (n_adv, n_val), final_acc, best_acc, best_epoch in summary:
        print(f"{pct:>5}% | {f'{n_adv} / {n_val}':>16} | {final_acc:>13.4f} | "
              f"{best_acc:>22.4f} | {best_epoch:>10}")

    fig, (ax_curves, ax_max) = plt.subplots(1, 2, figsize=(13, 5))

    for pct in sorted(sweep_history):
        accs = [h["val_acc"] for h in sweep_history[pct]]
        ax_curves.plot(range(1, len(accs) + 1), accs, marker="o", markersize=3, label=f"{pct}% adversarial")
    ax_curves.axvline(N_EPOCHS_TO_CHECK, color="grey", linestyle="--", linewidth=1)
    ax_curves.set_xlabel("Epoch")
    ax_curves.set_ylabel("Validation accuracy")
    ax_curves.set_title("Validation accuracy per epoch")
    ax_curves.legend()
    ax_curves.grid(True)

    ax_max.plot([s[0] for s in summary], [s[3] for s in summary], marker="o")
    ax_max.set_xlabel("Adversarial share of validation set (%)")
    ax_max.set_ylabel(f"Max validation accuracy (first {N_EPOCHS_TO_CHECK} epochs)")
    ax_max.set_title("Best validation accuracy vs adversarial share")
    ax_max.grid(True)

    plt.tight_layout()
    sweep_plot_path = "./sweep_val_accuracy.png"
    plt.savefig(sweep_plot_path)
    print(f"\nSweep plot saved to {sweep_plot_path}")
    plt.show()